# 03 - Split, Drift, and Feature Engineering

**Objectif :** créer un split de développement depuis le train brut, vérifier le leakage, analyser le drift et créer les features métier sans toucher au test final.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load raw train and keep final test untouched

In [ ]:
from credit_risk_lab.infrastructure import CreditRiskQualityChecker
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

raw_train_df = CsvLoanDataLoader(path=settings.raw_train_path).load()
raw_test_path = settings.raw_test_path

clean_train_df = CreditRiskQualityChecker().clean(raw_train_df)

{
    "clean_train_rows": len(clean_train_df),
    "reserved_final_test_path": str(raw_test_path),
}

## 2. Development train/validation split

In [ ]:
from credit_risk_lab.application import DatasetSplitter, SplitConfig

split_config = SplitConfig(
    test_size=settings.validation_size,
    random_state=settings.random_state,
    stratify=True,
)
splitter = DatasetSplitter(split_config)

model_train_df, validation_df = splitter.split(clean_train_df)
split_summary = splitter.summary(model_train_df, validation_df)
split_summary

## 3. Data leakage checks before preprocessing

In [ ]:
from credit_risk_lab.infrastructure.analytics import DataLeakageAuditor

leakage_auditor = DataLeakageAuditor(target_column=settings.target_column)
overlap_report = leakage_auditor.row_overlap_report(
    model_train_df,
    validation_df,
    holdout_name="validation",
)
target_report = leakage_auditor.target_leakage_report(
    model_train_df.drop(columns=[settings.target_column])
)

display(overlap_report)
target_report

## 4. Train vs validation drift analysis

In [ ]:
from credit_risk_lab.infrastructure.analytics import DriftAnalyzer

drift_analyzer = DriftAnalyzer(bins=10)
monitored_features = [
    column for column in model_train_df.columns if column != settings.target_column
]
drift_report = drift_analyzer.report_frame(
    model_train_df,
    validation_df,
    features=monitored_features,
)
drift_report.head(10)

## 5. Drift visualisation

In [ ]:
from credit_risk_lab.infrastructure.visualization import plot_drift_summary

plot_drift_summary(drift_report).show()

## 6. Business feature engineering on train and validation

In [ ]:
from credit_risk_lab.infrastructure.analytics import FeatureEngineeringReport
from credit_risk_lab.infrastructure.feature_engineering import LoanFeatureEngineer

feature_engineer = LoanFeatureEngineer()
featured_train_df = feature_engineer.transform(model_train_df)
featured_validation_df = feature_engineer.transform(validation_df)
feature_report = FeatureEngineeringReport(model_train_df, featured_train_df)
created_features = feature_report.created_columns()

print(f"Columns before feature engineering: {model_train_df.shape[1]}")
print(f"Columns after feature engineering: {featured_train_df.shape[1]}")
print(f"New columns created: {len(created_features)}")
print(f"Columns removed: {len(feature_report.removed_columns())}")
print(f"Retained columns modified: {len(feature_report.changed_columns())}")
print(f"Validation received the same column schema: {featured_train_df.columns.equals(featured_validation_df.columns)}")

display(feature_report.summary())
display(feature_report.created_columns_frame())
featured_train_df.head()

## 7. Fit preprocessing on train and transform validation

In [ ]:
from credit_risk_lab.infrastructure.modeling import CreditRiskPreprocessor

sensitive_columns = [
    column for column in settings.sensitive_columns if column in featured_train_df
]
target_column = settings.target_column

x_train = featured_train_df.drop(columns=[target_column, *sensitive_columns])
y_train = featured_train_df[target_column].reset_index(drop=True)
x_validation = featured_validation_df.drop(columns=[target_column, *sensitive_columns])
y_validation = featured_validation_df[target_column].reset_index(drop=True)

preprocessor = CreditRiskPreprocessor()
processed_train_features = preprocessor.fit_transform_frame(x_train)
processed_validation_features = preprocessor.transform_frame(x_validation)

processed_train_df = processed_train_features.assign(**{target_column: y_train.to_numpy()})
processed_validation_df = processed_validation_features.assign(
    **{target_column: y_validation.to_numpy()}
)

print(f"Columns before preprocessing: {x_train.shape[1]}")
print(f"Columns after preprocessing: {processed_train_features.shape[1]}")
print(f"Sensitive columns excluded: {sensitive_columns}")
print(f"Target excluded from features: {target_column}")
print(f"Preprocessor output features: {len(preprocessor.feature_names_)}")
print(f"Validation output columns match train: {processed_validation_features.columns.equals(processed_train_features.columns)}")

display(
    pd.DataFrame(
        [
            {"step": "after_feature_engineering", "train_columns": x_train.shape[1], "validation_columns": x_validation.shape[1]},
            {"step": "after_preprocessing", "train_columns": processed_train_features.shape[1], "validation_columns": processed_validation_features.shape[1]},
        ]
    )
)
display(pd.DataFrame({"model_feature": preprocessor.feature_names_}))

## 8. Persist transformed development artifacts

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CSVDatasetRepository

train_path = CSVDatasetRepository.save(processed_train_df, settings.train_path)
validation_path = CSVDatasetRepository.save(
    processed_validation_df,
    settings.validation_path,
)
preprocessor_path = preprocessor.save(settings.preprocessing_artifact_path)

{
    "processed_train_path": str(train_path),
    "processed_validation_path": str(validation_path),
    "preprocessor_path": str(preprocessor_path),
    "reserved_final_test_path": str(settings.raw_test_path),
}